# MCS to a small library search

This notebook uses SMSD 8.2.0 or later to turn a common graph into a SMARTS query, search a small molecular library, and explore fingerprint neighbours and clustering. Search results retain their completion and proof status. NumPy and scikit-learn are optional; the cells that need them report when they are missing.

## Setup

Run the next cell first in Colab or local Jupyter. It installs SMSD Pro and
the notebook's missing table or numerical dependencies into the current
kernel, using that kernel's Python. An already installed SMSD 8.2.0 or later
is kept. Installation needs internet access; the chemistry examples use inline
inputs. No wheel upload or copied setup cell is needed.

If an older SMSD is already imported, restart the kernel before running
setup. See [Installation](https://asad.github.io/SMSD/HOWTO-INSTALL/).


In [ ]:
# Run this cell first in Colab or a local Jupyter kernel.
import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Select a Python 3.12 or later kernel for SMSD Pro wheels")

try:
    _smsd_version = importlib.metadata.version("smsd")
except importlib.metadata.PackageNotFoundError:
    _smsd_version = "0.0.0"
_smsd_parts = tuple(int(part) for part in _smsd_version.split(".")[:3]
                    if part.isdigit())
_requirements = []
if importlib.util.find_spec("smsd") is None or _smsd_parts < (8, 2, 0):
    if "smsd" in sys.modules:
        raise RuntimeError("Restart the kernel, then run this setup cell before importing SMSD")
    _requirements.append("smsd>=8.2.0,<9")
if _requirements:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
         *_requirements], check=True,
    )
    importlib.invalidate_caches()
_optional_packages = {'numpy': 'numpy>=1.24,<3', 'sklearn': 'scikit-learn>=1.4,<2'}
_optional_requirements = [_package for _module, _package in _optional_packages.items()
                          if importlib.util.find_spec(_module) is None]
if _optional_requirements:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
             *_optional_requirements], check=True,
        )
    except subprocess.CalledProcessError:
        print("Some optional packages have no compatible wheel or could not be downloaded. "
              "The affected cells will report what is missing; SMSD remains available.")
    importlib.invalidate_caches()
import smsd
print("Using SMSD", smsd.__version__)


In [ ]:
# Responsive figures with a caption, an accessible image and an SVG fallback.
import base64
from html import escape


def show_svg(svg_text, description, max_width=800):
    image_uri = "data:image/svg+xml;base64," + base64.b64encode(
        svg_text.encode("utf-8")).decode("ascii")
    figure = (
        f'<figure style="margin:1em 0;max-width:{int(max_width)}px">'
        f'<img src="{image_uri}" alt="{escape(description, quote=True)}" '
        'style="display:block;width:100%;height:auto;box-sizing:border-box;'
        'background:#fff;border:1px solid #d5dce3;border-radius:8px;padding:16px">'
        f'<figcaption style="margin-top:0.6em;line-height:1.5">{escape(description)}</figcaption>'
        '</figure>'
    )
    try:
        from IPython.display import display
    except ImportError:
        print(description)
    else:
        display({"text/html": figure, "image/svg+xml": svg_text,
                 "text/plain": description}, raw=True,
                metadata={"image/svg+xml": {"alt": description}})


def show_table(frame):
    try:
        from IPython.display import display
    except ImportError:
        print(frame.to_string())
    else:
        display(frame)


In [ ]:
print("SMSD", smsd.__version__)


In [ ]:
import smsd

print("SMSD", smsd.__version__)
library = [
    "c1ccccc1O", "c1ccccc1N", "Cc1ccccc1", "CCO",
    "CCN", "CC(=O)O", "CCOc1ccccc1", "C1CCCCC1",
]
query_smiles, target_smiles = library[:2]

## Find the common graph and export SMARTS

The search below maximises common induced atoms under the default chemistry rules. Both the time and node limits are explicit. A found candidate is usable as a pattern even if stopped, but it is described as a maximum only when `optimal` is true. SMARTS export raises for matching rules that cannot be represented, rather than dropping those rules.

In [ ]:
options = smsd.MCS.Options()
options.timeout_ms = 1000
options.node_limit = 100000
common = smsd.MCS.find(query_smiles).with_options(options).match_with_smarts(target_smiles)
result = common.result

print({
    "found": result.found, "status": result.status.name,
    "optimal": result.optimal, "complete": result.complete,
    "nodes_visited": result.nodes_visited, "elapsed_ms": result.elapsed_ms,
})
if not result.found:
    raise RuntimeError("No common graph was found within the search budget")
pattern = common.smarts
print("Proved maximum pattern:" if result.optimal else "Candidate pattern; maximum unproved:", pattern)

In [ ]:
search_options = smsd.Substructure.Options()
search_options.timeout_ms = 1000
search_options.node_limit = 100000
search = smsd.Substructure.smarts(pattern).with_options(search_options)
search_results = search.match_many(library, num_threads=1)

for smiles, match in zip(library, search_results):
    print({
        "smiles": smiles, "found": match.found,
        "status": match.status.name, "optimal": match.optimal,
        "complete": match.complete,
    })
# An unfinished result with found=False does not prove absence.

In [ ]:
from smsd.highlight import mcs_svg

svg_text = mcs_svg(query_smiles, target_smiles, result, width=600, height=300)
try:
    from IPython.display import SVG, display
except ImportError:
    print("SVG generated as svg_text; inline display requires IPython")
else:
    show_svg(svg_text, "Query and target with the common substructure highlighted; "
             "the SVG title records search status and proof flags.")

## Optional fingerprint features

The feature adapter returns one native fingerprint per row. Binary features default to `uint8`; count features default to `int64`. The classifier below is a small pipeline demonstration with example labels, not an estimate of predictive performance.

In [ ]:
try:
    from smsd.adapters import fingerprints_numpy
    features = fingerprints_numpy(library, kind="ecfp", fp_size=256, num_threads=1)
except ImportError as error:
    print("NumPy features unavailable:", error)
else:
    print("Feature shape:", features.shape, "dtype:", features.dtype)
    print("Set bits per molecule:", features.sum(axis=1).tolist())

In [ ]:
try:
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.pipeline import make_pipeline
    from smsd.adapters import FingerprintTransformer
except ImportError as error:
    print("Optional scikit-learn pipeline unavailable:", error)
else:
    example_labels = [1, 1, 1, 0, 0, 0, 1, 0]
    model = make_pipeline(
        FingerprintTransformer(kind="ecfp", fp_size=256, num_threads=1),
        KNeighborsClassifier(n_neighbors=1),
    )
    model.fit(library, example_labels)
    print("Example predictions:", model.predict(["COc1ccccc1", "CCC"]).tolist())

## Packed corpus, top-k neighbours and clustering

The file stores packed binary fingerprints and is opened read-only through a memory map. Pair scans are exact and quadratic; thresholds or top-k limit the returned data. Butina groups molecules near each cluster centre, while MaxMin selects diverse rows. This example uses a temporary file that is removed on completion.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from smsd.corpus import FingerprintCorpus
from smsd.clustering import butina, maxmin

try:
    import numpy
except ImportError:
    print("Install the NumPy extra to run the mapped-corpus example")
else:
    with TemporaryDirectory() as directory:
        with FingerprintCorpus.from_smiles(
            Path(directory) / "library.fp", library,
            kind="ecfp", fp_size=256, chunk_size=3, num_threads=1,
        ) as corpus:
            neighbours = list(corpus.top_k_pairs(2, threshold=0.2, chunk_size=3))
            clusters = butina(corpus, threshold=0.5, reordering=True, chunk_size=3)
            diverse = maxmin(corpus, count=4, seed=0, chunk_size=3)
            print("Directed (query index, neighbour index, Tanimoto):", neighbours)
            print("Butina clusters (centre first):", clusters)
            print("MaxMin selection:", [(index, library[index]) for index in diverse])

All reported search statuses come from the engine. A fingerprint similarity or cluster assignment does not prove a chemical MCS. The temporary corpus stores bit width and input order; reuse the same fingerprint specification when supplying queries.